# 第六阶段：移动语义

## 实验 2：Rvalue Reference

实验 1 已经区分了 lvalue、xvalue 和 prvalue。本实验继续回答：`T&&` 能绑定什么、为什么命名的 rvalue reference 仍是 lvalue，以及获得一个右值引用是否就意味着资源已经移动。

完成后应能解释：

- `T&&` 是引用，不会创建第二个 `T` 对象；
- 普通 rvalue reference 只能直接绑定 rvalue；
- 任何有名字的变量表达式都是 lvalue，即使变量类型是 `T&&`；
- `std::move` 只允许后续操作考虑移动，不负责搬运资源；
- rvalue reference 参数本身不等于 ownership transfer contract。

In [1]:
// 本步骤：引入断言、输出、智能指针、字符串和类型检查所需的标准库。
#include <cassert>
#include <iostream>
#include <memory>
#include <string>
#include <string_view>
#include <type_traits>
#include <utility>

### 1. `T&&` 绑定 rvalue，但它本身只是别名

普通的 `int&&` 可以绑定临时值，也可以绑定由 `std::move` 产生的 xvalue。它不能直接绑定 lvalue。引用只为现有对象提供另一个访问入口，不拥有独立存储。

In [2]:
// 本步骤：分别把 rvalue reference 绑定到临时对象和现有对象的 xvalue。
{
    // 临时 int 的生命周期被 temporary_reference 延长到当前作用域结束。
    int &&temporary_reference = 42;

    int value = 10;

    // std::move(value) 仍指向 value；moved_reference 不会创建或搬运 int。
    int &&moved_reference = std::move(value);

    // 通过两个名字修改的是各自引用所指向的原对象。
    temporary_reference += 1;
    moved_reference += 5;

    assert(temporary_reference == 43);
    assert(value == 15);
    assert(&moved_reference == &value);

    std::cout << "temporary value: " << temporary_reference << '\n'
              << "same address: " << std::boolalpha
              << (&moved_reference == &value) << '\n';
}

temporary value: 43
same address: true


下面的代码不能编译，应保留为阅读示例：

```cpp
int value = 10;
int &&reference = value;
// error: rvalue reference cannot bind to lvalue
```

如果调用方明确允许把 `value` 当作可移动对象使用，应写成 `int&& reference = std::move(value);`。

### 2. 引用类型与表达式类别是两个问题

变量的声明类型可以是 `int&&`，但只要在表达式中写出它的名字，这个表达式就是 lvalue。原因不是它的内容不能移动，而是这个名字能稳定定位一个对象。

In [3]:
// 本步骤：定义引用重载，用实际重载结果观察命名 rvalue reference 的类别。
std::string_view binding_kind(int &value) noexcept
{
    // 参数只用于选择重载，不读取具体数值。
    static_cast<void>(value);
    return "lvalue binding";
}

std::string_view binding_kind(int &&value) noexcept
{
    // 该重载接收 prvalue 或 xvalue。
    static_cast<void>(value);
    return "rvalue binding";
}

In [4]:
// 本步骤：对比 reference 与 std::move(reference) 的类型推导和重载选择。
{
    int &&reference = 42;

    // 无额外括号的 decltype 读取变量的声明类型。
    static_assert(std::is_same_v<decltype(reference), int &&>);

    // 加括号后检查表达式：有名字的 reference 是 lvalue。
    static_assert(std::is_same_v<decltype((reference)), int &>);
    assert(binding_kind(reference) == "lvalue binding");

    // std::move(reference) 把 lvalue expression 转成 xvalue。
    static_assert(std::is_same_v<decltype((std::move(reference))), int &&>);
    assert(binding_kind(std::move(reference)) == "rvalue binding");

    std::cout << "reference: " << binding_kind(reference) << '\n'
              << "move(reference): "
              << binding_kind(std::move(reference)) << '\n';
}

reference: lvalue binding
move(reference): rvalue binding


可以把规则压缩为：

```text
声明：int&& reference = 42;

decltype(reference)               -> int&& （声明类型）
decltype((reference))             -> int&  （lvalue expression）
decltype((std::move(reference)))  -> int&& （xvalue expression）
```

### 3. rvalue reference 可以延长临时对象生命周期

临时对象直接绑定到局部 rvalue reference 时，其生命周期会延长到该引用所在作用域结束。这个规则避免引用立即悬空，但不会让引用变成可复制的 owner。

In [5]:
// 本步骤：定义可观察构造和析构的对象，验证临时对象的生命周期延长。
class LifetimeProbe
{
public:
    explicit LifetimeProbe(std::string name)
        : name_(std::move(name))
    {
        std::cout << "construct: " << name_ << '\n';
    }

    ~LifetimeProbe() noexcept
    {
        std::cout << "destroy: " << name_ << '\n';
    }

    const std::string &name() const noexcept
    {
        // 返回对象内部字符串的只读借用。
        return name_;
    }

private:
    std::string name_;
};

In [6]:
// 本步骤：把临时对象绑定到局部 rvalue reference，并观察析构发生在作用域末尾。
{
    std::cout << "enter scope\n";

    // 临时 LifetimeProbe 不会在本条语句结束时立即销毁。
    LifetimeProbe &&probe = LifetimeProbe("temporary");

    // 这里仍可安全借用临时对象内部的 name。
    assert(probe.name() == "temporary");
    std::cout << "inside scope: " << probe.name() << '\n';

    // probe 所绑定的对象将在右花括号处析构。
    std::cout << "leave scope next\n";
}

enter scope
construct: temporary
inside scope: temporary
leave scope next
destroy: temporary


生命周期延长只适用于特定的直接绑定情形。返回指向局部对象的 `T&&` 仍会悬空：

```cpp
LifetimeProbe&& broken()
{
    LifetimeProbe local("local");
    return std::move(local); // local 在函数返回时销毁
}
```

引用不能让被引用对象活得超过原本的 owner。

### 4. 绑定 rvalue reference 不等于已经移动

`std::move(source)` 和 `std::string&& alias` 都只改变访问方式。真正的资源转移发生在某个 move constructor 或 move assignment 消费这个 xvalue 时。

In [7]:
// 本步骤：先建立右值引用别名，再调用字符串移动构造，区分 cast 与实际移动。
{
    std::string source = "native buffer";
    const std::string original = source;

    // alias 与 source 指向同一个字符串；此时尚未调用移动构造。
    std::string &&alias = std::move(source);
    assert(&alias == &source);
    assert(source == original);

    // 用 xvalue 初始化新对象，此处才会调用 std::string 的移动构造。
    std::string destination = std::move(alias);
    assert(destination == original);

    // moved-from source 仍然有效，但内容只保证满足 string 的有效状态约束。
    std::cout << "destination: " << destination << '\n'
              << "source size after move: " << source.size() << '\n';

    // 对 moved-from 对象执行赋值是合法的，可重新建立明确状态。
    source = "reused";
    assert(source == "reused");
}

destination: native buffer
source size after move: 0


不要依赖 moved-from `std::string` 一定为空。标准库只保证它仍是有效对象；除非具体类型另有说明，最稳妥的操作是销毁它、重新赋值，或调用明确允许的无前置条件操作。

### 5. 参数写成 `T&&` 也不自动取得 ownership

函数参数类型只表达它接受哪类表达式。函数是否真正消费资源，仍由函数实现和 API contract 决定。下面先让 `observe_text` 只读借用字符串，再让 `consume_owner` 按值接收 `unique_ptr`，明确转移 ownership。

In [8]:
// 本步骤：定义只观察 rvalue 的接口，以及真正接收唯一 ownership 的接口。
void observe_text(std::string &&text)
{
    // text 是命名参数，因此函数体内是 lvalue；这里只读取，不转移资源。
    std::cout << "observed: " << text << '\n';
}

void consume_owner(std::unique_ptr<int> owner)
{
    // 按值参数成为新的唯一 owner，并在函数结束时自动释放对象。
    assert(owner != nullptr);
    std::cout << "consumed value: " << *owner << '\n';
}

In [9]:
// 本步骤：对比“允许绑定 rvalue”与“把 ownership 交给函数”两种 contract。
{
    std::string text = "borrow only";

    // observe_text 接受 xvalue，但实现没有移动 text 的资源。
    observe_text(std::move(text));
    assert(text == "borrow only");

    auto owner = std::make_unique<int>(42);

    // unique_ptr 按值传参会调用移动构造，把唯一 ownership 交给参数。
    consume_owner(std::move(owner));

    // 调用结束后资源已释放，原 unique_ptr 保持可析构的空状态。
    assert(owner == nullptr);
}

observed: borrow only
consumed value: 42


### 6. 普通 rvalue reference 与 forwarding reference 的边界

本实验讨论的 `Widget&&`、`std::string&&` 是普通 rvalue reference。下面这种依赖模板类型推导的 `T&&` 是 forwarding reference，可以同时接收 lvalue 和 rvalue：

```cpp
template <typename T>
void forward_to_core(T&& value);
```

它依赖 reference collapsing 与 `std::forward`，将在后续实验单独展开。不能看到所有 `T&&` 都直接解释为“只接收右值”。

### 7. Native SDK、C ABI 与 Kotlin/Native

rvalue reference 是 C++ 内部的表达式与重载机制，不属于稳定 C ABI。不能把下面的接口直接暴露给 Kotlin/Native：

```cpp
void submit(Buffer&& buffer);
```

跨 ABI 边界应把 ownership transfer 翻译成明确的 C contract，例如：

```c
NativeStatus native_submit(NativeBufferHandle* buffer);
// 成功后 handle 被消费；失败时 ownership 是否保留必须写入文档。
```

Kotlin 包装层再用类、`close()` 或受约束的状态封装 handle。关键不是把 `&&` 翻译成另一种语法，而是保留“谁在成功或失败后拥有资源”的 contract。

## 实验结论

看到 `T&&` 时应依次判断：

1. 它是普通 rvalue reference，还是模板推导产生的 forwarding reference？
2. 当前使用的是变量的声明类型，还是这个变量名表达式的 value category？
3. 这里只是建立别名，还是确实调用了移动构造或移动赋值？
4. API 是临时借用对象，还是明确消费 ownership？

核心规则是：

```text
T&& variable        -> 变量类型是 rvalue reference
variable            -> 命名表达式是 lvalue
std::move(variable) -> xvalue，只提供移动机会
move operation      -> 才可能真正转移资源
```